In [1]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import corner

In [ ]:

def sigma_ext(lam, n, k, V_np, n_m):
    return ((18 * np.pi * (n_m)**(3/2)) / (lam)) * V_np * (k / ((n+2*n_m)**2 + k**2))

def absorbance(l, rho, lam, n, k, V_np, n_m):
    sigma = sigma_ext(lam, n, k, V_np, n_m)
    return -np.log10(np.exp(-sigma * rho * l))

def log_likelihood(theta, lam, absorbance_data, n, k, n_m):
    V_np, rho, l, sigma = theta

    prediction = absorbance(l, rho, lam, n, k, V_np, n_m)
    
    gaussian = -0.5 * np.sum(np.log(((absorbance_data - prediction) / sigma) ** 2) + np.log(2 * np.pi * sigma ** 2))
    return gaussian

def log_prior(theta):
    V_np, rho, l, sigma = theta

    V_np_interval = (0, 1e-20)
    rho_interval = (0, 1e20)
    l_interval = (0, 2e-7)
    sigma_interval = (0, 10)

    if V_np_interval[0] < V_np < V_np_interval[1] and rho_interval[0] < rho < rho_interval[1] and l_interval[0] < l < l_interval[1] and sigma_interval[0] < sigma < sigma_interval[1]:
        return 0.0
    return -np.inf

def log_posterior(theta, lam, absorbance_data, n, k, n_m):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, lam, absorbance_data, n, k, n_m)


